In [4]:
import subprocess
import time

In [5]:
def setup():
    !pip install -q langchain_community langchain_ollama langchain_chroma pypdf rank_bm25 langchain_core flashrank google-colab
    !apt-get -qq update && apt-get -qq install -y zstd
    !curl -fsSL https://ollama.com/install.sh | sh
    
    subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/tmp/ollama.log", "w"),
        stderr=subprocess.STDOUT
    )
    time.sleep(5)
    
    !ollama pull qwen3-embedding:0.6b

setup()

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.8 requires langchain-core<2.0.0,>=1.4.4, but you have langchain-core 0.3.86 which is incompatible.
langchain-classic 1.0.8 requires langchain-text-splitters<2.0.0,>=1.1.2, but you have langchain-text-splitters 0.3.11 which is incompatible.
langgraph 1.2.11 requires langchain-core<2,>=1.4.7, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.86 which

In [3]:
!nvidia-smi # pour s'assurer que le GPU de Google colab marche bien

Sat Sep 12 17:20:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Dans ce fichier on va traiter les points suivants:

- charger les documents

- Fusionner les pages en un seul texte
- diviser les données selon "Article" (Chunking) et ajouter les métadonnées
- embeddings
- enregistrer les données dans chromaDB

<hr style="border: 1px solid white; width: 80%;">


In [4]:
# Les bibliothéques utilisées

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

import re 

/tmp/ipykernel_701/1555652351.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### 1. Charger les données

In [5]:
#pdf_loader_fr = PyPDFLoader(file_path= r"..\data\raw\Code du travail.pdf")

pdf_loader_fr = PyPDFLoader(file_path= "Code du travail.pdf") 
pages = pdf_loader_fr.load()


print(f"\nLe nombre de pages est {len(pages)}") # doit etre 206!



Le nombre de pages est 206


### 2. Fusionner toutes les pages en un seul texte

In [6]:
type(pages)

list

In [7]:
full_text = "\n".join([page.page_content for page in pages]) # on sépare chaque page et la suivante par "\n"


full_text

'CODE DU TRAVAIL  \nVersion consolidée en date du 26 octobre 2011\n-2 -  \nLOI N° 65.99 FORMANT CODE DU \nTRAVAIL1 \n \n \nComme il a été modifié : \n- Dahir 1.11.170 du 27 kaada 1432 (25 octobre 2011) bulletin \nofficiel en arabe n°5989 bis du 28 kaada 1432 (26 octobre 2011 ), \npage 5228 portant promulgation de la loi n°58.11 relative à la \ncour de cassation et modifiant le dahir n°1.57.223 du 2 Rabii I \n1377 (27 septembre 1957) relatif à la cour suprême. \n- Dahir n°1.06.233 du 28 rabii I 1428 (17 avril 2007) portant \npromulgation de la loi n°48.06 portant suppression de service \nmilitaire, bulletin officiel n°5522 du 15 rabii II 1428 (3 -5-2007), \npage 581. \n \n \n \n \n \n \n \n \n   \n                                           \n1 - Bulletin Officiel n°5210 du 16 Rabii 1425 (6 Mai 2004), p.600.\n-3 -  \nDAHIR N° 1-03-194 DU 14 REJEB 1424 (11 \nSEPTEMBRE 2003) PORTANT \nPROMULGATION DE LA LOI N° 65-99 \nRELATIVE AU CODE DU TRAVAIL. \n \nLOUANGE A DIEU SEUL  \n \n(Grand Sceau

In [8]:
type(full_text)

str

### 3. Chunking et ajout des méta-données

In [9]:
# On cherche le mot Article suivi de "premier" ou d'un numéro quelconque

# r"(\nArticle\s*(?:\n?\s*premier|\d+))"

pattern = r"(\nArticle\s*(?:premier|\d+))"   
parts = re.split(pattern, full_text)



chunks = [] 

for i in range(1, len(parts), 2):
    title = parts[i].strip()     # Article premier , Article 2, Article 3 , .... Article 589   
    content = parts[i+1].strip()    # Le contenu de chaque article
    
    
    
    full_article_text = f"{title}\n{content}"
    clean_title = re.sub(r"\s+", " ", title) # on remplace espace (si il existe) par " "
    
    
    chunks.append(Document(
        page_content=full_article_text,
        metadata={"article_no": clean_title, "source": "Code du Travail Marocain"}
    ))

print(f"Nombre total de chunks créés : {len(chunks)}")

Nombre total de chunks créés : 588


In [10]:
type(chunks)

list

In [11]:
chunks[0].page_content

"Article premier\nLes dispositions de la présente loi s'appliquent aux personnes liées \npar un contrat de travail quels que soient ses modalités d'exécution, la \nnature de la rémunération et le mode de son paiement qu'il prévoit et la \nnature de l'entreprise dans laquelle il s'exécute, notamment les \nentreprises industrielles, commerciales, artisanales et les exploitations \nagricoles et forestières et leurs dépendances. Elles s'appliquent \négalement aux entreprises et établissements à caractère industriel, \ncommercial ou agricole relevant de l'Etat et des collectivités locales, aux \ncoopératives, sociétés civiles, syndicats, associations et groupements de \ntoute nature. \nLes dispositions de la présente loi s'appliquent également aux \nemployeurs exerçant une profession libérale, au secteur des services et, \nde manière générale, aux personnes liées par un contrat de travail dont \nl'activité ne relève d'aucune de celles précitées."

Après avoir créer les chunks, on va nettoyer le texte en remplaçant "\n" et les doubles espaces par " ".

In [12]:
# Remplacer \n et les doubles espaces par " " dans chque chunk 
for chunk in chunks:
    chunk.page_content = re.sub(r"\n", " ", chunk.page_content)
    chunk.page_content = re.sub(r"\s+", " ", chunk.page_content).strip()

In [13]:
type(chunks)

list

In [14]:
chunks[0].page_content

"Article premier Les dispositions de la présente loi s'appliquent aux personnes liées par un contrat de travail quels que soient ses modalités d'exécution, la nature de la rémunération et le mode de son paiement qu'il prévoit et la nature de l'entreprise dans laquelle il s'exécute, notamment les entreprises industrielles, commerciales, artisanales et les exploitations agricoles et forestières et leurs dépendances. Elles s'appliquent également aux entreprises et établissements à caractère industriel, commercial ou agricole relevant de l'Etat et des collectivités locales, aux coopératives, sociétés civiles, syndicats, associations et groupements de toute nature. Les dispositions de la présente loi s'appliquent également aux employeurs exerçant une profession libérale, au secteur des services et, de manière générale, aux personnes liées par un contrat de travail dont l'activité ne relève d'aucune de celles précitées."

### 4. Embeddings

In [15]:
# Initialisation du modèle embedding
embedding_model = OllamaEmbeddings(
    model= "qwen3-embedding:0.6b"
)



In [16]:
type(embedding_model)

langchain_ollama.embeddings.OllamaEmbeddings

### 5. enregistrer les données dans chromaDB

In [17]:
from google.colab import drive
drive.mount('/content/drive')

persist_directory = '/content/drive/MyDrive/Chatbot_Loi_Travail/chroma_db'



vector_store = Chroma.from_documents(
    documents= chunks,
    embedding= embedding_model,
    #persist_directory='./chroma_db'
    persist_directory= persist_directory
)

Mounted at /content/drive


In [18]:
type(vector_store)

langchain_chroma.vectorstores.Chroma

#### Tester que les données ont été bien enregistrer dans ChromaDB

In [19]:
# Récupérer le nombre total d'éléments dans la collection ChromaDB
count = vector_store._collection.count()

print(f"Nombre d'articles stockés dans ChromaDB : {count}") #doit etre 588 == len(chunks)


Nombre d'articles stockés dans ChromaDB : 588


In [20]:
# Inspecter un document et ses métadonnées dans chromadb

data = vector_store._collection.get(limit= 1)

print("--- Métadonnées : ---")
print(data['metadatas'])

print("--- Contenu : ---")
print(data['documents'][0][:200])


--- Métadonnées : ---
[{'article_no': 'Article premier', 'source': 'Code du Travail Marocain'}]
--- Contenu : ---
Article premier Les dispositions de la présente loi s'appliquent aux personnes liées par un contrat de travail quels que soient ses modalités d'exécution, la nature de la rémunération et le mode de so
